# Comprehensive Evaluation

Reads `all_predictions_clean.parquet` (saved by the cleanup notebook) and produces the full visual and tabular evaluation suite.

**Figures produced** (saved to `models/figures/`):

1. Pre vs post cleanup headline metrics (showing the format-artifact remediation had minimal impact)
2. Per-bucket recall heatmap — the headline diagnostic, ordered by difficulty
3. Recall vs FPR — production operating curve, zoomed to deployment-relevant region
4. Score distributions per model, split by class
5. Calibration / reliability diagrams
6. Model agreement pies — which errors are shared vs idiosyncratic
7. Positive recall by source (MCQ vs prose rephrasings)
8. Error counts by source bucket

**Tabular outputs**:

- Error summary statistics per model
- FP/FN breakdown by source bucket
- Universally-misclassified examples (saved to CSV)
- Consolidated results table (saved to CSV)

CPU is sufficient; ~2 minutes end-to-end.

## Environment setup

In [ ]:
!pip install -q scikit-learn pandas pyarrow matplotlib

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=True)
DATA_DIR = Path("/content/drive/MyDrive/biology_refusal/data")
OUT_DIR  = Path("/content/drive/MyDrive/biology_refusal/models")
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR:  {OUT_DIR}")

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (
    classification_report, confusion_matrix, precision_recall_curve,
    average_precision_score, roc_auc_score, f1_score, roc_curve,
)

# Visual style — clean, consistent across all figures
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 160,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

MODEL_COLORS = {
    "Model 1": "#4A90D9",   # blue
    "Model 2": "#E89B3C",   # orange
    "Model 3": "#5BB17A",   # green
}
MODEL_SHORT = {
    "Model 1": "TF-IDF",
    "Model 2": "MiniLM+LR",
    "Model 3": "DistilBERT",
}

FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)
print(f"Saving figures to: {FIG_DIR}")

# Load cleaned-data predictions written by the cleanup notebook
preds_df = pd.read_parquet(OUT_DIR / "all_predictions_clean.parquet")
print(f"Loaded {len(preds_df)} test rows with predictions from all 3 models")
print(f"Source breakdown:\n{preds_df['source'].value_counts()}")

# Pre-cleanup baseline numbers (from the modeling notebook output) — used for
# the delta panel. Hardcoded here because we did not save pre-cleanup state.
PRE_CLEANUP = {
    "Model 1": {"f1_pos": 0.943, "f1_neg": 0.983, "ap": 0.986, "auc": 0.995},
    "Model 2": {"f1_pos": 0.936, "f1_neg": 0.980, "ap": 0.980, "auc": 0.994},
    "Model 3": {"f1_pos": 0.975, "f1_neg": 0.993, "ap": 0.995, "auc": 0.998},
}

# Map model column names to canonical labels
MODEL_COLS = {
    "Model 1": ("model_1_pred", "model_1_score"),
    "Model 2": ("model_2_pred", "model_2_score"),
    "Model 3": ("model_3_pred", "model_3_score"),
}

y_true = preds_df["label"].to_numpy()

## Figure 1 — Pre vs post cleanup metrics

In [ ]:
# Bar chart per model showing aggregate F1, AP, ROC-AUC before and after the
# leakage-cleanup pass. Conveys the headline finding: cleanup barely changed
# absolute numbers, meaning models were mostly learning content, not provenance.

post_metrics = {}
for model in MODEL_COLS:
    pred_col, score_col = MODEL_COLS[model]
    y_pred  = preds_df[pred_col].to_numpy()
    y_score = preds_df[score_col].to_numpy()
    post_metrics[model] = {
        "f1_pos": f1_score(y_true, y_pred, pos_label=1),
        "f1_neg": f1_score(y_true, y_pred, pos_label=0),
        "ap":     average_precision_score(y_true, y_score),
        "auc":    roc_auc_score(y_true, y_score),
    }

metric_names = ["f1_pos", "f1_neg", "ap", "auc"]
metric_labels = ["F1 (refuse)", "F1 (don't refuse)", "AP", "ROC-AUC"]

fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharey=True)
x = np.arange(3)
width = 0.38

for ax, m, label in zip(axes, metric_names, metric_labels):
    pre  = [PRE_CLEANUP[mod][m] for mod in MODEL_COLS]
    post = [post_metrics[mod][m] for mod in MODEL_COLS]
    bars1 = ax.bar(x - width/2, pre,  width, label="Pre-cleanup",  color="#B0B0B0", edgecolor="black", linewidth=0.5)
    bars2 = ax.bar(x + width/2, post, width, label="Post-cleanup", color="#4A90D9", edgecolor="black", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_COLS], fontsize=9)
    ax.set_title(label)
    ax.set_ylim(0.85, 1.005)
    ax.grid(axis="y", alpha=0.3)
    # Annotate deltas
    for i, (p, q) in enumerate(zip(pre, post)):
        delta = q - p
        color = "darkgreen" if delta > 0 else ("darkred" if delta < -0.005 else "gray")
        ax.text(i, max(p, q) + 0.005, f"{delta:+.3f}", ha="center", fontsize=8, color=color, fontweight="bold")

axes[0].set_ylabel("Score")
axes[0].legend(loc="lower right", fontsize=9)
fig.suptitle("Headline Metrics: Pre- vs Post-Cleanup\n"
             "Small deltas mean format-detection wasn't load-bearing — models were learning content all along",
             fontsize=12, y=1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / "01_pre_post_metrics.png", bbox_inches="tight")
plt.show()
print("Saved 01_pre_post_metrics.png")

## Figure 2 — Per-bucket recall heatmap

In [ ]:
# Visualizes how each model performs on each negative bucket (the headline
# diagnostic for content vs format discrimination).

bucket_recall = {}
for model, (pred_col, _) in MODEL_COLS.items():
    bucket_recall[model] = {}
    for bucket in preds_df[preds_df["label"] == 0]["source"].unique():
        sub = preds_df[preds_df["source"] == bucket]
        recall = (sub[pred_col] == 0).mean()
        bucket_recall[model][bucket] = recall

# Order buckets by difficulty (alpaca easiest -> dual_use hardest)
bucket_order = ["alpaca", "mmlu_other", "mmlu_bio", "dual_use"]
matrix = np.array([
    [bucket_recall[m][b] for b in bucket_order]
    for m in MODEL_COLS
])

fig, ax = plt.subplots(figsize=(9, 4))
# Custom colormap: red (bad) -> yellow -> green (good), with the color stops
# tuned so 0.95+ looks clearly green and <0.85 clearly red
cmap = LinearSegmentedColormap.from_list(
    "rg", ["#D9534F", "#F0AD4E", "#5CB85C"], N=256,
)
im = ax.imshow(matrix, cmap=cmap, vmin=0.75, vmax=1.0, aspect="auto")

ax.set_xticks(np.arange(len(bucket_order)))
ax.set_xticklabels(bucket_order, fontsize=10)
ax.set_yticks(np.arange(len(MODEL_COLS)))
ax.set_yticklabels([MODEL_SHORT[m] for m in MODEL_COLS], fontsize=10)
ax.set_xlabel("Negative bucket (left = easiest, right = hardest)")
ax.set_title("Per-Bucket Recall on Negatives — The Headline Diagnostic\n"
             "Recall = fraction of bucket correctly classified as 'don't refuse'", fontsize=11)

# Annotations with both the value and the support size
bucket_n = {b: (preds_df["source"] == b).sum() for b in bucket_order}
for i in range(len(MODEL_COLS)):
    for j, bucket in enumerate(bucket_order):
        val = matrix[i, j]
        n = bucket_n[bucket]
        color = "white" if val < 0.92 else "black"
        ax.text(j, i, f"{val:.3f}\n(n={n})", ha="center", va="center",
                color=color, fontsize=10, fontweight="bold")

# Difficulty gradient annotation
ax.annotate("difficulty →", xy=(3.4, -0.7), xytext=(-0.4, -0.7),
            arrowprops=dict(arrowstyle="->", color="gray"),
            ha="center", color="gray", fontsize=9, annotation_clip=False)

cbar = plt.colorbar(im, ax=ax, shrink=0.7, label="Recall")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_per_bucket_recall.png", bbox_inches="tight")
plt.show()
print("Saved 02_per_bucket_recall.png")

## Figure 3 — Recall vs FPR (the production-relevance plot)

In [ ]:
# At deployment, you'd choose a threshold by acceptable false-positive rate
# on benign traffic. This plot shows the recall-FPR tradeoff across the full
# threshold range for each model — the most production-relevant comparison.

fig, ax = plt.subplots(figsize=(9, 6))

for model, (pred_col, score_col) in MODEL_COLS.items():
    y_score = preds_df[score_col].to_numpy()
    fpr, tpr, _ = roc_curve(y_true, y_score)
    ax.plot(fpr, tpr, lw=2, label=f"{MODEL_SHORT[model]}",
            color=MODEL_COLORS[model])

# Shade the production-relevant operating zone
ax.axvspan(0, 0.01, alpha=0.15, color="green", label="FPR ≤ 0.01 (tight)")
ax.axvspan(0.01, 0.05, alpha=0.10, color="goldenrod", label="FPR ≤ 0.05 (moderate)")

# Vertical reference lines at the canonical FPR thresholds
for fpr_target, ls in [(0.01, ":"), (0.05, "--"), (0.10, "-.")]:
    ax.axvline(fpr_target, color="gray", lw=0.8, ls=ls, alpha=0.5)

ax.set_xlim(0, 0.2)  # zoom into low-FPR region where deployment cares
ax.set_ylim(0, 1.01)
ax.set_xlabel("False Positive Rate (benign traffic falsely refused)")
ax.set_ylabel("Recall (positives correctly refused)")
ax.set_title("Recall vs. FPR — Production Operating Curve (low-FPR region)\n"
             "Shaded zones show realistic deployment thresholds", fontsize=11)
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.3)

# Annotate recall at FPR=0.01 for each model
for model, (_, score_col) in MODEL_COLS.items():
    y_score = preds_df[score_col].to_numpy()
    fpr, tpr, _ = roc_curve(y_true, y_score)
    idx = np.argmin(np.abs(fpr - 0.01))
    rec_at_1pct = tpr[idx]
    ax.scatter([0.01], [rec_at_1pct], s=80, zorder=5,
               color=MODEL_COLORS[model], edgecolor="black", linewidth=1.2)
    ax.annotate(f"{rec_at_1pct:.2f}", (0.01, rec_at_1pct),
                xytext=(8, -3), textcoords="offset points",
                fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(FIG_DIR / "03_recall_vs_fpr.png", bbox_inches="tight")
plt.show()
print("Saved 03_recall_vs_fpr.png")

## Figure 4 — Score distribution histograms (calibration story)

In [ ]:
# Shows the predicted-probability distribution for each model, split by true
# label. Reveals calibration character — DistilBERT's smoothly graded scores
# vs. the linear models' pushed-to-extremes scores.

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
bins = np.linspace(0, 1, 41)

for ax, (model, (_, score_col)) in zip(axes, MODEL_COLS.items()):
    y_score = preds_df[score_col].to_numpy()
    neg_scores = y_score[y_true == 0]
    pos_scores = y_score[y_true == 1]
    ax.hist(neg_scores, bins=bins, alpha=0.6, color="#4A90D9",
            label=f"don't refuse (n={len(neg_scores)})", edgecolor="black", linewidth=0.3)
    ax.hist(pos_scores, bins=bins, alpha=0.6, color="#D9534F",
            label=f"refuse (n={len(pos_scores)})", edgecolor="black", linewidth=0.3)
    ax.axvline(0.5, color="black", ls="--", lw=1, alpha=0.6, label="threshold=0.5")
    ax.set_xlabel("Predicted P(refuse)")
    ax.set_title(MODEL_SHORT[model])
    ax.legend(fontsize=8, loc="upper center")
    ax.set_yscale("log")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Count (log scale)")
fig.suptitle("Score Distributions by True Label\n"
             "Well-separated histograms = good discrimination. "
             "Mass at intermediate values = calibrated probabilities.",
             fontsize=12, y=1.05)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_score_distributions.png", bbox_inches="tight")
plt.show()
print("Saved 04_score_distributions.png")

## Figure 5 — Reliability (calibration) diagrams

In [ ]:
# Bins predictions by predicted probability, plots empirical accuracy per bin.
# Perfect calibration = diagonal line. Models above the diagonal are under-
# confident; below are over-confident.

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
n_bins = 10

for ax, (model, (_, score_col)) in zip(axes, MODEL_COLS.items()):
    y_score = preds_df[score_col].to_numpy()
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_acc = []
    bin_count = []
    for i in range(n_bins):
        mask = (y_score >= bin_edges[i]) & (y_score < bin_edges[i+1])
        if i == n_bins - 1:  # include upper boundary in last bin
            mask = (y_score >= bin_edges[i]) & (y_score <= bin_edges[i+1])
        if mask.sum() > 0:
            bin_acc.append(y_true[mask].mean())
            bin_count.append(mask.sum())
        else:
            bin_acc.append(np.nan)
            bin_count.append(0)
    bin_acc = np.array(bin_acc)
    bin_count = np.array(bin_count)

    # Bar width scaled by bin count (log scale visually)
    sizes = 20 + 300 * (bin_count / max(bin_count.max(), 1))
    ax.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5, label="perfect calibration")
    valid = ~np.isnan(bin_acc)
    ax.scatter(bin_centers[valid], bin_acc[valid], s=sizes[valid],
               color=MODEL_COLORS[model], alpha=0.7, edgecolor="black", linewidth=1)
    ax.plot(bin_centers[valid], bin_acc[valid], color=MODEL_COLORS[model], lw=1.5, alpha=0.6)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel("Predicted P(refuse)")
    ax.set_title(MODEL_SHORT[model])
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(alpha=0.3)
    ax.set_aspect("equal")

axes[0].set_ylabel("Empirical fraction of refuse")
fig.suptitle("Reliability Diagrams (Calibration)\n"
             "Marker size ∝ examples per bin. Points on the diagonal = well-calibrated.",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "05_calibration.png", bbox_inches="tight")
plt.show()
print("Saved 05_calibration.png")

## Figure 6 — Model agreement matrix

In [ ]:
# For each test example, count how many models got it wrong. Pie chart per
# class. Tells us whether models fail on the same examples (intrinsically hard)
# or different examples (each picks up different signals).

n_models = len(MODEL_COLS)
err_count = np.zeros(len(preds_df), dtype=int)
for model, (pred_col, _) in MODEL_COLS.items():
    err_count += (preds_df[pred_col].to_numpy() != y_true).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pie for negatives (class 0)
neg_mask = y_true == 0
neg_err_dist = Counter(err_count[neg_mask])
labels_neg = [f"{k} models wrong\n(n={neg_err_dist[k]})" for k in sorted(neg_err_dist)]
sizes_neg  = [neg_err_dist[k] for k in sorted(neg_err_dist)]
colors_neg = ["#5CB85C", "#F0AD4E", "#E89B3C", "#D9534F"][:len(sizes_neg)]
axes[0].pie(sizes_neg, labels=labels_neg, autopct=lambda p: f"{p:.1f}%" if p > 1 else "",
            colors=colors_neg, startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[0].set_title(f"Negatives (n={neg_mask.sum()})\nHow many models miss each example?")

# Pie for positives (class 1)
pos_mask = y_true == 1
pos_err_dist = Counter(err_count[pos_mask])
labels_pos = [f"{k} models wrong\n(n={pos_err_dist[k]})" for k in sorted(pos_err_dist)]
sizes_pos  = [pos_err_dist[k] for k in sorted(pos_err_dist)]
colors_pos = ["#5CB85C", "#F0AD4E", "#E89B3C", "#D9534F"][:len(sizes_pos)]
axes[1].pie(sizes_pos, labels=labels_pos, autopct=lambda p: f"{p:.1f}%" if p > 1 else "",
            colors=colors_pos, startangle=90, wedgeprops={"edgecolor": "white", "linewidth": 1.5})
axes[1].set_title(f"Positives (n={pos_mask.sum()})\nHow many models miss each example?")

fig.suptitle("Model Agreement on Errors\n"
             "Large 'all 3 wrong' slice = intrinsically hard examples. "
             "Large 'just 1 wrong' = models exploit different signals.",
             fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "06_model_agreement.png", bbox_inches="tight")
plt.show()
print("Saved 06_model_agreement.png")

# Print the numbers
print("\nModel agreement on errors:")
print(f"  Negatives (n={neg_mask.sum()}):")
for k in sorted(neg_err_dist):
    print(f"    {k} models wrong: {neg_err_dist[k]:4d} ({100*neg_err_dist[k]/neg_mask.sum():.1f}%)")
print(f"  Positives (n={pos_mask.sum()}):")
for k in sorted(pos_err_dist):
    print(f"    {k} models wrong: {pos_err_dist[k]:4d} ({100*pos_err_dist[k]/pos_mask.sum():.1f}%)")

## Figure 7 — Per-source positive recall comparison

In [ ]:
# Specifically compares recall on original WMDP stems vs rephrased prose-form
# versions. Tests whether format-augmentation generalized.

pos_recall = {}
for model, (pred_col, _) in MODEL_COLS.items():
    pos_recall[model] = {}
    for src in preds_df[preds_df["label"] == 1]["source"].unique():
        sub = preds_df[preds_df["source"] == src]
        pos_recall[model][src] = (sub[pred_col] == 1).mean()

src_order = ["wmdp_bio_original", "wmdp_bio_rephrased"]
fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(src_order))
width = 0.27
for i, model in enumerate(MODEL_COLS):
    vals = [pos_recall[model][s] for s in src_order]
    ax.bar(x + (i-1)*width, vals, width, color=MODEL_COLORS[model],
           edgecolor="black", linewidth=0.5, label=MODEL_SHORT[model])
    for j, v in enumerate(vals):
        ax.text(x[j] + (i-1)*width, v + 0.005, f"{v:.2f}",
                ha="center", fontsize=8, fontweight="bold")

src_n = {s: (preds_df["source"] == s).sum() for s in src_order}
ax.set_xticks(x)
ax.set_xticklabels([f"{s}\n(n={src_n[s]})" for s in src_order])
ax.set_ylabel("Recall (positives correctly identified)")
ax.set_ylim(0.85, 1.02)
ax.set_title("Positive Recall: Original MCQ vs Programmatic Prose Rephrasings\n"
             "Tests whether format-augmentation generalized across question forms",
             fontsize=11)
ax.legend(loc="lower right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "07_positive_recall_by_source.png", bbox_inches="tight")
plt.show()
print("Saved 07_positive_recall_by_source.png")

## Error analysis — categorize failure modes

In [ ]:
# Pull misclassifications per model, compute summary statistics by:
#   - source bucket (where do errors concentrate?)
#   - score confidence (are wrong predictions confident or borderline?)

print("\n" + "=" * 70)
print("  ERROR ANALYSIS: WHERE DO MODELS FAIL?")
print("=" * 70)

err_summary = []
for model, (pred_col, score_col) in MODEL_COLS.items():
    y_pred = preds_df[pred_col].to_numpy()
    y_score = preds_df[score_col].to_numpy()
    err_mask = y_pred != y_true
    fp_mask = (y_pred == 1) & (y_true == 0)
    fn_mask = (y_pred == 0) & (y_true == 1)

    # Confident vs borderline (score in [0.2, 0.8] = borderline)
    confident_err = err_mask & ((y_score >= 0.8) | (y_score <= 0.2))
    borderline_err = err_mask & (y_score > 0.2) & (y_score < 0.8)

    err_summary.append({
        "model": MODEL_SHORT[model],
        "total_errors": err_mask.sum(),
        "false_positives": fp_mask.sum(),
        "false_negatives": fn_mask.sum(),
        "confident_errors": confident_err.sum(),
        "borderline_errors": borderline_err.sum(),
    })

err_summary_df = pd.DataFrame(err_summary)
print("\n", err_summary_df.to_string(index=False))

# Breakdown of FPs by source bucket per model
print("\nFalse positives by source bucket (predicted refuse, actually shouldn't):")
fp_by_source = pd.DataFrame()
for model, (pred_col, _) in MODEL_COLS.items():
    y_pred = preds_df[pred_col].to_numpy()
    fps = preds_df[(y_pred == 1) & (y_true == 0)]
    fp_by_source[MODEL_SHORT[model]] = fps["source"].value_counts()
fp_by_source = fp_by_source.fillna(0).astype(int)
print("\n", fp_by_source.to_string())

# Breakdown of FNs by source per model
print("\nFalse negatives by source (predicted don't refuse, actually should):")
fn_by_source = pd.DataFrame()
for model, (pred_col, _) in MODEL_COLS.items():
    y_pred = preds_df[pred_col].to_numpy()
    fns = preds_df[(y_pred == 0) & (y_true == 1)]
    fn_by_source[MODEL_SHORT[model]] = fns["source"].value_counts()
fn_by_source = fn_by_source.fillna(0).astype(int)
print("\n", fn_by_source.to_string())

## Figure 8 — Error count visualization

In [ ]:
# Stacked bar of FPs and FNs per model, broken down by source bucket.

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Stack order — keep consistent for legend readability
neg_buckets = ["alpaca", "mmlu_other", "mmlu_bio", "dual_use"]
pos_buckets = ["wmdp_bio_original", "wmdp_bio_rephrased"]
bucket_palette = {
    "alpaca":     "#90C695",
    "mmlu_other": "#FFD27F",
    "mmlu_bio":   "#F4A582",
    "dual_use":   "#C5566B",
    "wmdp_bio_original":  "#7B68EE",
    "wmdp_bio_rephrased": "#B19CD9",
}

# False positives panel
ax = axes[0]
x = np.arange(len(MODEL_COLS))
bottom = np.zeros(len(MODEL_COLS))
for b in neg_buckets:
    vals = [fp_by_source.loc[b, MODEL_SHORT[m]] if b in fp_by_source.index else 0
            for m in MODEL_COLS]
    ax.bar(x, vals, bottom=bottom, label=b, color=bucket_palette[b],
           edgecolor="black", linewidth=0.5)
    for i, v in enumerate(vals):
        if v > 0:
            ax.text(i, bottom[i] + v/2, str(int(v)), ha="center", va="center",
                    fontsize=8, fontweight="bold")
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_COLS])
ax.set_title("False Positives by Source\n(falsely refused benign prompts)")
ax.set_ylabel("Count")
ax.legend(title="Source bucket", loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.3)

# False negatives panel
ax = axes[1]
bottom = np.zeros(len(MODEL_COLS))
for b in pos_buckets:
    vals = [fn_by_source.loc[b, MODEL_SHORT[m]] if b in fn_by_source.index else 0
            for m in MODEL_COLS]
    ax.bar(x, vals, bottom=bottom, label=b, color=bucket_palette[b],
           edgecolor="black", linewidth=0.5)
    for i, v in enumerate(vals):
        if v > 0:
            ax.text(i, bottom[i] + v/2, str(int(v)), ha="center", va="center",
                    fontsize=8, fontweight="bold")
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels([MODEL_SHORT[m] for m in MODEL_COLS])
ax.set_title("False Negatives by Source\n(missed refuse prompts)")
ax.legend(title="Source bucket", loc="upper right", fontsize=8)
ax.grid(axis="y", alpha=0.3)

fig.suptitle("Error Counts by Model and Source", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIG_DIR / "08_errors_by_source.png", bbox_inches="tight")
plt.show()
print("Saved 08_errors_by_source.png")

## Hardest examples — what ALL models missed

In [ ]:
# Extract test rows where all three models classified incorrectly. These are
# the intrinsically hard examples and what the write-up's qualitative section
# should reference (without quoting specific content).

print("\n" + "=" * 70)
print("  EXAMPLES MISSED BY ALL THREE MODELS")
print("=" * 70)

err_count = np.zeros(len(preds_df), dtype=int)
for model, (pred_col, _) in MODEL_COLS.items():
    err_count += (preds_df[pred_col].to_numpy() != y_true).astype(int)

# All three wrong
all_wrong_mask = err_count == 3
all_wrong = preds_df[all_wrong_mask].copy()
print(f"\nTotal examples ALL 3 models got wrong: {len(all_wrong)}")
if len(all_wrong) > 0:
    print(f"  Breakdown by label and source:")
    print(all_wrong.groupby(["label", "source"]).size().to_string())

    # Save for the write-up
    all_wrong.to_csv(OUT_DIR / "all_models_wrong.csv", index=False)
    print(f"\n  Saved to {OUT_DIR}/all_models_wrong.csv")

# Examples where exactly 2 of 3 disagreed with the label (interesting middle ground)
two_wrong_mask = err_count == 2
two_wrong = preds_df[two_wrong_mask].copy()
print(f"\nExamples where exactly 2 of 3 models disagreed: {len(two_wrong)}")
if len(two_wrong) > 0:
    print(two_wrong.groupby(["label", "source"]).size().to_string())

## Bonus — comparative confidence on hardest examples

In [ ]:
# For the examples ALL models got wrong: what scores did they assign?
# Tells us whether the errors are confident-wrong (concerning) or
# borderline-uncertain (more forgivable).

if len(all_wrong) > 0:
    print("\nScore profile of universally-misclassified examples:")
    print(f"  (label=1 means refuse, scores closer to 0 = model said 'don't refuse')")
    score_cols = [score_col for _, score_col in MODEL_COLS.values()]
    confidence_summary = all_wrong[["label", "source"] + score_cols].copy()
    confidence_summary.columns = ["label", "source", "TF-IDF", "MiniLM+LR", "DistilBERT"]
    print(confidence_summary.to_string(index=False))

## Consolidated results table

In [ ]:
# Final consolidated table summarizing all metrics across models.

print("\n" + "=" * 70)
print("  CONSOLIDATED RESULTS TABLE (for write-up)")
print("=" * 70)

final_table = pd.DataFrame(index=[MODEL_SHORT[m] for m in MODEL_COLS])
for model in MODEL_COLS:
    pred_col, score_col = MODEL_COLS[model]
    y_pred  = preds_df[pred_col].to_numpy()
    y_score = preds_df[score_col].to_numpy()

    final_table.loc[MODEL_SHORT[model], "Accuracy"] = (y_pred == y_true).mean()
    final_table.loc[MODEL_SHORT[model], "F1 (refuse)"] = f1_score(y_true, y_pred, pos_label=1)
    final_table.loc[MODEL_SHORT[model], "F1 (don't refuse)"] = f1_score(y_true, y_pred, pos_label=0)
    final_table.loc[MODEL_SHORT[model], "AP"] = average_precision_score(y_true, y_score)
    final_table.loc[MODEL_SHORT[model], "ROC-AUC"] = roc_auc_score(y_true, y_score)

    # Per-bucket recall
    for bucket in bucket_order:
        sub = preds_df[preds_df["source"] == bucket]
        recall = (sub[pred_col] == 0).mean()
        final_table.loc[MODEL_SHORT[model], f"Rec@{bucket}"] = recall

    # Recall at FPR=0.01
    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_score)
    idx = np.argmin(np.abs(fpr_arr - 0.01))
    final_table.loc[MODEL_SHORT[model], "Rec@FPR=1%"] = tpr_arr[idx]

print("\n", final_table.round(3).to_string())
final_table.round(4).to_csv(OUT_DIR / "results_table.csv")
print(f"\nSaved consolidated table to {OUT_DIR}/results_table.csv")

print("\n" + "=" * 70)
print(f"  ALL FIGURES SAVED TO: {FIG_DIR}")
print("=" * 70)
print("\nFigures generated:")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  - {f.name}")